# ASV-Visualizer

Notebook untuk memvisualisasikan hasil **MitoNGS** (tabel ASV + hasil taksonomi):

1. **Krona** (interaktif, HTML)
2. **Sankey** (interaktif, HTML + PNG)
3. **Stacked bar plot** untuk **Class**, **Order**, dan **Family**

**Input** (taruh di folder `sample/`): salah satu dari  
- **satu file gabungan** MitoNGS, misalnya `ASVTable (5).tsv.gz` (taksonomi + read per sampel; `.gz` langsung bisa dibaca), atau  
- `asv_table.tsv` + `taxresults.csv` (dua file terpisah)  

**Output**: `result/` (HTML, tabel CSV) dan `result/figures/` (PNG/SVG)

Jalankan sel dari atas ke bawah (*Run > Run All Cells*). Pengaturan ada di sel nomor 1.

## 1. Pengaturan & import

In [ ]:
import io, os, re, sys, gzip, fnmatch, shutil, subprocess, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.graph_objects as go
import plotly.express as px

warnings.filterwarnings("ignore")

# ------------------------- PATH -------------------------
BASE_DIR   = Path.cwd()                      # folder tempat notebook ini berada
SAMPLE_DIR = BASE_DIR / "sample"

def find_file(patterns):
    """Cari file di sample/ (huruf besar/kecil diabaikan). Nama file bebas, mis. 'ASVTable (5).tsv.gz'."""
    files = sorted(p for p in SAMPLE_DIR.glob("*") if p.is_file())
    for pat in patterns:
        for p in files:
            if fnmatch.fnmatch(p.name.lower(), pat.lower()):
                return p
    return None

ASV_FILE = find_file(["asv_table.*", "asvtable*", "*.tsv.gz", "*.tsv"])
TAX_FILE = find_file(["taxresults.*", "*taxresults*"])      # opsional; kosong jika taksonomi sudah ada di ASV table
RESULT_DIR = BASE_DIR / "result"
FIG_DIR    = RESULT_DIR / "figures"
RESULT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------- PENGATURAN -------------------------
# Kolom taksonomi. None = deteksi otomatis. Isi manual jika deteksi gagal, contoh:
#   TAX_ID_COL      = "ASV_ID"
#   RANK_COLS       = {"phylum": "Phylum", "class": "Class", "order": "Order", "family": "Family"}
#   LINEAGE_STR_COL = "Taxonomy"      # jika taksonomi ada dalam 1 kolom: "Chordata;Aves;Psittaciformes;..."
TAX_ID_COL      = None
RANK_COLS       = {}
LINEAGE_STR_COL = None

TOP_N            = 15        # jumlah taksa teratas di stacked bar (sisanya -> "Others")
RELATIVE         = True      # True = kelimpahan relatif (%), False = jumlah read mentah
SANKEY_RANKS     = ["phylum", "class", "order", "family"]   # tingkat taksonomi di Sankey
SANKEY_MIN_PCT   = 0.5       # node < x% dari total read digabung jadi "Others" (kecilkan jika ingin lebih detail)
SAMPLE_ORDER     = None      # contoh: ["S1", "S2", "S3"]; None = urutan seperti di asv_table
EXPORT_SVG       = True      # simpan stacked bar juga dalam format SVG
AMBIGUOUS        = "unresolved"   # hit ganda di 1 rank (mis. "A;B"): "unresolved" = tak teridentifikasi di rank itu, "first" = ambil nama pertama
CLEAN_SAMPLE_NAMES = True    # "3.fastq" -> "3"

plt.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})
print("Folder kerja :", BASE_DIR)
print("ASV table    :", ASV_FILE if ASV_FILE else "TIDAK DITEMUKAN di folder sample/")
print("Tax results  :", TAX_FILE if TAX_FILE else "(tidak ada) -> memakai taksonomi di dalam ASV table")

## 2. Membaca data

In [ ]:
RANKS = ["domain", "kingdom", "phylum", "class", "order", "family", "genus", "species"]
ALIASES = {
    "domain":  {"domain", "superkingdom"},
    "kingdom": {"kingdom"},
    "phylum":  {"phylum", "division"},
    "class":   {"class"},
    "order":   {"order"},
    "family":  {"family"},
    "genus":   {"genus"},
    "species": {"species"},
}
MISSING = {"", "nan", "none", "na", "n/a", "null", "-", "unassigned", "unclassified", "unknown",
           "no hit", "no_hit", "nohit", "not assigned"}


SAMPLE_EXCLUDE = {"top_hits", "tophits", "percent_identity", "identity", "pident", "query_coverage", "coverage",
                  "qcov", "e_value", "evalue", "bitscore", "bit_score", "length", "sequence", "seq", "md5",
                  "taxonomy", "confidence"}
RANK_WORDS = set().union(*ALIASES.values())


def open_text(path):
    if str(path).lower().endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8-sig")
    return open(path, encoding="utf-8-sig")


def read_asv_table(path):
    """Baca tabel ASV (baris = ASV, kolom pertama = ID, kolom numerik = sampel).
    Kolom taksonomi / identity / E-value / sequence otomatis dipisah sebagai metadata."""
    if path is None or not Path(path).exists():
        raise FileNotFoundError("File ASV table tidak ditemukan di folder sample/.\n"
                                "Letakkan file (mis. 'ASVTable (5).tsv.gz' atau 'asv_table.tsv') di sana lalu jalankan ulang.")
    with open_text(path) as fh:
        lines = [l for l in fh.read().splitlines() if l.strip() and not l.startswith("# ")]
    lines[0] = lines[0].lstrip("#").strip()
    sep = "\t" if "\t" in lines[0] else ("," if "," in lines[0] else ";")
    df = pd.read_csv(io.StringIO("\n".join(lines)), sep=sep, low_memory=False)
    df = df.rename(columns={df.columns[0]: "ASV"})
    df["ASV"] = df["ASV"].astype(str).str.strip()
    df = df.drop_duplicates("ASV").set_index("ASV")

    sample_cols, extra_cols = [], []
    for c in df.columns:
        low = str(c).strip().lower()
        if low in SAMPLE_EXCLUDE or low in RANK_WORDS:
            extra_cols.append(c); continue
        num = pd.to_numeric(df[c], errors="coerce")
        if num.notna().mean() >= 0.9:
            df[c] = num.fillna(0)
            sample_cols.append(c)
        else:
            extra_cols.append(c)
    counts = df[sample_cols].astype(float)
    counts = counts.loc[:, counts.sum() > 0]          # buang sampel kosong
    if CLEAN_SAMPLE_NAMES:
        new = [re.sub(r"\.(fastq|fq)(\.gz)?$", "", str(c), flags=re.I) for c in counts.columns]
        if len(set(new)) == len(new):
            counts.columns = new
    return counts, df[extra_cols]


def clean_name(x):
    """Bersihkan nama taksa. Nilai kosong / 'unclassified' / hit ganda ("A;B") -> NaN (tak teridentifikasi)."""
    if pd.isna(x):
        return np.nan
    parts = []
    for p in re.split(r"\s*;\s*", str(x)):
        s = re.sub(r"^[a-zA-Z]__", "", p.strip()).strip().strip("'\"")
        if s and s.lower() not in MISSING and not s.lower().startswith(("unclassified", "unassigned")):
            parts.append(s)
    parts = list(dict.fromkeys(parts))                 # unik, urutan dipertahankan
    if not parts:
        return np.nan
    if len(parts) == 1:
        return parts[0]
    return parts[0] if AMBIGUOUS == "first" else np.nan


def find_rank_columns(tax):
    found = {}
    for rank, names in ALIASES.items():
        exact = [c for c in tax.columns if str(c).strip().lower() in names]
        if exact:
            found[rank] = exact[0]; continue
        loose = [c for c in tax.columns
                 if names & set(re.split(r"[^a-z]+", str(c).strip().lower()))]
        if loose:
            found[rank] = loose[0]
    return found


def split_lineage(series):
    """Pecah kolom lineage tunggal (dipisah ; | atau ,) menjadi kolom rank."""
    parts = series.astype(str).str.split(r"\s*[;|]\s*", regex=True, expand=True)
    if parts.shape[1] == 1:
        parts = series.astype(str).str.split(r"\s*,\s*", regex=True, expand=True)
    return parts


def read_taxonomy(path, asv_ids, embedded=None):
    """Taksonomi dari taxresults (jika ada), kalau tidak dari kolom taksonomi di dalam ASV table (embedded)."""
    if path is not None and Path(path).exists():
        tax = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig", dtype=str, compression="infer")
        print("Sumber taksonomi       :", Path(path).name)
    elif embedded is not None and len(embedded.columns) > 0:
        tax = embedded.reset_index().astype(str).replace("nan", np.nan)
        print("Sumber taksonomi       : kolom di dalam ASV table")
    else:
        raise FileNotFoundError("Tidak ada taksonomi: taxresults.csv tidak ada dan ASV table tidak punya kolom taksonomi.")
    tax.columns = [str(c).strip() for c in tax.columns]

    # --- kolom ID: pilih kolom yang paling banyak cocok dengan ID ASV ---
    asv_set = set(asv_ids)
    if TAX_ID_COL:
        id_col = TAX_ID_COL
    else:
        overlap = {c: len(asv_set & set(tax[c].astype(str).str.strip())) for c in tax.columns}
        id_col = max(overlap, key=overlap.get)
        if overlap[id_col] == 0:
            raise ValueError("Tidak ada kolom di taxresults.csv yang cocok dengan ID ASV di asv_table.\n"
                             f"Kolom tersedia: {list(tax.columns)}\nSet TAX_ID_COL secara manual di sel pengaturan.")
    tax["ASV"] = tax[id_col].astype(str).str.strip()
    tax = tax.drop_duplicates("ASV").set_index("ASV")

    # --- kolom rank ---
    cols = dict(RANK_COLS) if RANK_COLS else find_rank_columns(tax.drop(columns=[id_col], errors="ignore"))
    out = pd.DataFrame(index=tax.index)
    if cols:
        for rank, c in cols.items():
            out[rank] = tax[c]
    else:
        lin_col = LINEAGE_STR_COL
        if lin_col is None:
            cand = [c for c in tax.columns if re.search(r"taxon|lineage|taxonomy", c, re.I)]
            lin_col = cand[0] if cand else None
        if lin_col is None:
            raise ValueError("Kolom rank (Phylum/Class/Order/Family) tidak terdeteksi.\n"
                             f"Kolom tersedia: {list(tax.columns)}\nIsi RANK_COLS atau LINEAGE_STR_COL di sel pengaturan.")
        parts = split_lineage(tax[lin_col])
        # petakan posisi ke rank; jika ada prefiks (p__, c__) pakai prefiks
        prefix_map = {"d": "domain", "k": "kingdom", "p": "phylum", "c": "class", "o": "order",
                      "f": "family", "g": "genus", "s": "species"}
        for i in range(parts.shape[1]):
            col = parts.iloc[:, i]
            m = col.dropna().astype(str).str.extract(r"^([a-zA-Z])__")[0].dropna()
            if len(m) and m.mode().iloc[0].lower() in prefix_map:
                out[prefix_map[m.mode().iloc[0].lower()]] = col
        if out.shape[1] == 0:                       # tanpa prefiks: asumsi urutan standar
            std = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]
            for i in range(min(parts.shape[1], len(std))):
                out[std[i]] = parts.iloc[:, i]

    ranks = [r for r in RANKS if r in out.columns]
    raw = out[ranks].copy()                                  # teks asli (sebelum dibersihkan)
    out = out[ranks].apply(lambda s: s.map(clean_name))
    return out, ranks, id_col, cols, raw


def fill_lineage(df, ranks):
    """Isi rank kosong: 'Unclassified <taksa induk terakhir>' atau 'Unassigned' bila tidak ada sama sekali."""
    out = df[ranks].copy().astype(object)
    last = pd.Series("", index=out.index, dtype=object)
    for r in ranks:
        known = out[r].notna()
        filler = pd.Series(np.where(last == "", "Unassigned", "Unclassified " + last.astype(str)), index=out.index)
        new_last = out[r].where(known, last)
        out[r] = out[r].where(known, filler)
        last = new_last
    return out


counts, asv_extra = read_asv_table(ASV_FILE)
if SAMPLE_ORDER:
    counts = counts[[s for s in SAMPLE_ORDER if s in counts.columns]]
sample_cols = list(counts.columns)

tax_raw, RANKS_AVAIL, id_col, rank_map, tax_text = read_taxonomy(TAX_FILE, counts.index, embedded=asv_extra)
tax_filled = fill_lineage(tax_raw.reindex(counts.index), RANKS_AVAIL)
merged = counts.join(tax_filled)

print(f"Kolom ID di taxresults : {id_col}")
print(f"Rank terdeteksi        : {RANKS_AVAIL}")
print(f"Jumlah ASV             : {len(counts):,}")
print(f"Jumlah sampel          : {len(sample_cols)}  -> {sample_cols}")

### Ringkasan data (QC singkat)

In [ ]:
n_asv_tax = tax_raw.reindex(counts.index).notna().any(axis=1).sum()
summary = pd.DataFrame({
    "Total read": counts.sum().astype(int),
    "Jumlah ASV (>0)": (counts > 0).sum(),
})
for r in ["class", "order", "family"]:
    if r in RANKS_AVAIL:
        assigned = counts[tax_raw.reindex(counts.index)[r].notna().values].sum() / counts.sum() * 100
        summary[f"% read teridentifikasi ({r})"] = assigned.round(1)
display(summary)

# hit ganda ("A;B") per rank = ASV yang cocok ke >1 taksa -> dianggap tak teridentifikasi di rank itu
amb_rows = []
for r in RANKS_AVAIL:
    multi = tax_text.reindex(counts.index)[r].map(
        lambda v: isinstance(v, str) and len({p.strip() for p in v.split(";") if p.strip()}) > 1).values
    if multi.any():
        amb_rows.append({"rank": r, "ASV hit ganda": int(multi.sum()),
                         "% read": round(counts[multi].sum().sum() / counts.sum().sum() * 100, 1)})
if amb_rows:
    print(f"Hit ganda (AMBIGUOUS = '{AMBIGUOUS}'):")
    display(pd.DataFrame(amb_rows).set_index("rank"))
print(f"ASV dengan taksonomi: {n_asv_tax:,} dari {len(counts):,} ({n_asv_tax/len(counts)*100:.1f}%)")
display(merged.head())

## 3. Krona

Jika **KronaTools** (`ktImportText`) terpasang, dihasilkan `result/krona.html` (satu dataset per sampel + `ALL_SAMPLES`).  
Jika tidak (misalnya di Windows), notebook otomatis membuat **sunburst Plotly** (`result/krona_like_sunburst.html`) yang tampilannya mirip Krona.

In [ ]:
def safe_name(s):
    return re.sub(r"[^\w\-.]+", "_", str(s))

krona_ranks = [r for r in RANKS_AVAIL]
krona_dir = RESULT_DIR / "krona_input"
krona_dir.mkdir(exist_ok=True)
for f in krona_dir.glob("*.txt"):
    f.unlink()

pooled = merged[sample_cols].sum(axis=1)
pooled_df = (merged[krona_ranks].assign(count=pooled.values).query("count > 0")
             .groupby(krona_ranks)["count"].sum().reset_index())
datasets = {"ALL_SAMPLES": pooled} if len(sample_cols) > 1 else {}
datasets.update({s: merged[s] for s in sample_cols})

krona_args = []
for i, (name, series) in enumerate(datasets.items()):
    t = merged[krona_ranks].copy()
    t["count"] = series.values
    t = t[t["count"] > 0].groupby(krona_ranks)["count"].sum().reset_index()
    fpath = krona_dir / f"{i:02d}_{safe_name(name)}.txt"
    with open(fpath, "w", encoding="utf-8") as fh:
        for _, row in t.iterrows():
            fh.write(f"{row['count']:.0f}\t" + "\t".join(str(row[r]) for r in krona_ranks) + "\n")
    krona_args.append(f"{fpath},{safe_name(name)}")

kt = shutil.which("ktImportText")
if kt is None:                                    # cari di folder environment aktif
    for cand in [Path(sys.prefix) / "bin" / "ktImportText", Path(sys.prefix) / "Scripts" / "ktImportText"]:
        if cand.exists():
            kt = str(cand); break

krona_html = RESULT_DIR / "krona.html"
if kt:
    proc = subprocess.run([kt, "-o", str(krona_html), "-n", "ASV"] + krona_args, capture_output=True, text=True)
    if proc.returncode == 0 and krona_html.exists():
        print("Krona berhasil dibuat :", krona_html)
        print("Buka di browser       :", krona_html.resolve().as_uri())
    else:
        print("ktImportText gagal:\n", proc.stderr[:800]); kt = None

if not kt:
    print("KronaTools tidak ditemukan / gagal -> membuat sunburst Plotly sebagai pengganti.")
    fig_sun = px.sunburst(pooled_df, path=krona_ranks, values="count", maxdepth=4)
    fig_sun.update_traces(textinfo="label+percent root", insidetextorientation="radial")
    fig_sun.update_layout(width=850, height=850, margin=dict(t=30, l=10, r=10, b=10),
                          title="Komposisi taksonomi (semua sampel)")
    out = RESULT_DIR / "krona_like_sunburst.html"
    fig_sun.write_html(out, include_plotlyjs=True)
    try:
        fig_sun.write_image(str(FIG_DIR / "krona_like_sunburst.png"), width=1000, height=1000, scale=2)
    except Exception as e:
        print("(PNG sunburst dilewati:", type(e).__name__, ")")
    print("Tersimpan:", out)
    fig_sun.show()

## 4. Sankey

Alur kelimpahan (total read semua sampel) antar tingkat taksonomi (`SANKEY_RANKS`).  
Node di bawah `SANKEY_MIN_PCT` % digabung menjadi **Others**.

In [ ]:
def hex_rgba(color, alpha=0.4):
    r, g, b = mcolors.to_rgb(color)
    return f"rgba({int(r*255)},{int(g*255)},{int(b*255)},{alpha})"


def make_palette(n):
    cmaps = [plt.get_cmap(n_) for n_ in ("tab20", "tab20b", "tab20c")]
    cols = [mcolors.to_hex(cm(i)) for cm in cmaps for i in range(cm.N)]
    cols = [c for c in cols if mcolors.rgb_to_hsv(mcolors.to_rgb(c))[1] > 0.25]   # buang warna abu-abu
    return [cols[i % len(cols)] for i in range(n)]


def build_sankey(merged, sample_cols, ranks, min_pct):
    lin = merged[ranks].copy()
    lin["reads"] = merged[sample_cols].sum(axis=1)
    lin = lin[lin["reads"] > 0].groupby(ranks)["reads"].sum().reset_index()
    total = lin["reads"].sum()
    lin = lin.astype(object)

    # gabungkan node kecil -> "Others" (dan hentikan jalur di tingkat itu)
    for i, r in enumerate(ranks):
        active = lin[r].notna()
        if not active.any():
            continue
        tot = lin[active].groupby(ranks[:i + 1])["reads"].transform("sum").astype(float)
        small = pd.Series(False, index=lin.index)
        small[active] = (tot / total * 100) < min_pct
        lin.loc[small, r] = "Others"
        for deeper in ranks[i + 1:]:
            lin.loc[small, deeper] = None

    node_index, labels, node_level, node_root = {}, [], [], []
    links = {}
    def get_node(level, path):
        key = (level, path)
        if key not in node_index:
            node_index[key] = len(labels)
            labels.append(path[-1]); node_level.append(level); node_root.append(path[0])
        return node_index[key]

    for _, row in lin.iterrows():
        path, prev = (), None
        for lvl, r in enumerate(ranks):
            val = row[r]
            if val is None or (isinstance(val, float) and np.isnan(val)):
                break
            path = path + (val,)
            cur = get_node(lvl, path)
            if prev is not None:
                links[(prev, cur)] = links.get((prev, cur), 0) + float(row["reads"])
            prev = cur
    return labels, node_level, node_root, links, total


sk_ranks = [r for r in SANKEY_RANKS if r in RANKS_AVAIL]
missing = [r for r in SANKEY_RANKS if r not in RANKS_AVAIL]
if missing:
    print(f"Peringatan: rank {missing} tidak ada di data, dilewati.")
assert len(sk_ranks) >= 2, "Sankey butuh minimal 2 rank."

labels, node_level, node_root, links, total = build_sankey(merged, sample_cols, sk_ranks, SANKEY_MIN_PCT)

roots = list(dict.fromkeys(node_root))
pal = dict(zip(roots, make_palette(len(roots))))
for k in roots:
    if k == "Others":
        pal[k] = "#9e9e9e"
    elif k.startswith("Unclassified") or k == "Unassigned":
        pal[k] = "#c7c7c7"
node_colors = [pal[r] for r in node_root]

src = [s for (s, t) in links]; tgt = [t for (s, t) in links]; val = list(links.values())
fig_sk = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(label=labels, color=node_colors, pad=14, thickness=18,
              line=dict(color="white", width=0.5),
              hovertemplate="%{label}<br>%{value:,.0f} read<extra></extra>"),
    link=dict(source=src, target=tgt, value=val,
              color=[hex_rgba(node_colors[s]) for s in src],
              hovertemplate="%{source.label} → %{target.label}<br>%{value:,.0f} read<extra></extra>"),
))
n = len(sk_ranks)
for i, r in enumerate(sk_ranks):
    fig_sk.add_annotation(x=i / (n - 1), y=1.07, xref="paper", yref="paper", showarrow=False,
                          text=f"<b>{r.capitalize()}</b>", font=dict(size=14),
                          xanchor="left" if i == 0 else ("right" if i == n - 1 else "center"))
height = int(max(700, 14 * max(node_level.count(l) for l in range(n)) + 200))
fig_sk.update_layout(title=f"Sankey komposisi taksonomi (semua sampel; node < {SANKEY_MIN_PCT}% digabung)",
                     font=dict(size=11), width=1300, height=height, margin=dict(t=110, l=20, r=20, b=20))

fig_sk.write_html(RESULT_DIR / "sankey.html", include_plotlyjs=True)
print("Tersimpan:", RESULT_DIR / "sankey.html")
try:
    fig_sk.write_image(str(FIG_DIR / "sankey.png"), width=1600, height=int(height * 1.2), scale=2)
    print("Tersimpan:", FIG_DIR / "sankey.png")
except Exception as e:
    print("PNG Sankey dilewati (kaleido belum berfungsi):", type(e).__name__, "- gunakan tombol kamera di plot HTML.")
fig_sk.show()

## 5. Stacked bar plot (Class, Order, Family)

Default: kelimpahan **relatif (%)** dengan `TOP_N` taksa teratas, sisanya = **Others**.  
Atur `TOP_N` / `RELATIVE` di sel pengaturan. Tabel angka juga disimpan di `result/`.

In [ ]:
def stacked_bar(rank, top_n=TOP_N, relative=RELATIVE):
    if rank not in RANKS_AVAIL:
        print(f"Rank '{rank}' tidak ada di data, dilewati."); return
    tbl = merged.groupby(rank)[sample_cols].sum()
    tbl = tbl.loc[tbl.sum(axis=1).sort_values(ascending=False).index]
    tbl.to_csv(RESULT_DIR / f"abundance_{rank}.csv")

    if len(tbl) > top_n:
        top = tbl.iloc[:top_n].copy()
        top.loc["Others"] = tbl.iloc[top_n:].sum()
        tbl = top
    plot_tbl = tbl / tbl.sum() * 100 if relative else tbl

    names = list(plot_tbl.index)
    base = make_palette(len(names))
    colors = []
    for i, nm in enumerate(names):
        if nm == "Others":
            colors.append("#9e9e9e")
        elif nm.startswith("Unclassified") or nm == "Unassigned":
            colors.append("#d9d9d9")
        else:
            colors.append(base[i])

    single = len(sample_cols) == 1                      # 1 sampel -> batang ramping
    fig, ax = plt.subplots(figsize=(7.5, 6.5) if single else (max(8, 0.6 * len(sample_cols) + 5), 6.5))
    plot_tbl.T.plot(kind="bar", stacked=True, ax=ax, color=colors, width=0.4 if single else 0.85,
                    edgecolor="white", linewidth=0.4)
    if single:
        ax.set_xlim(-0.9, 0.9)
    ax.set_ylabel("Kelimpahan relatif (%)" if relative else "Jumlah read")
    ax.set_xlabel("")
    ax.set_title(f"Komposisi taksonomi tingkat {rank.capitalize()}", fontsize=13, fontweight="bold")
    if relative:
        ax.set_ylim(0, 100)
    plt.setp(ax.get_xticklabels(), rotation=0 if single else 45, ha="center" if single else "right")
    ax.legend(title=rank.capitalize(), bbox_to_anchor=(1.01, 1), loc="upper left",
              fontsize=8, title_fontsize=9, frameon=False)
    fig.tight_layout()

    fig.savefig(FIG_DIR / f"stacked_bar_{rank}.png", dpi=300, bbox_inches="tight")
    if EXPORT_SVG:
        fig.savefig(FIG_DIR / f"stacked_bar_{rank}.svg", bbox_inches="tight")
    print("Tersimpan:", FIG_DIR / f"stacked_bar_{rank}.png")
    plt.show()


stacked_bar("class")

In [ ]:
stacked_bar('order')

In [ ]:
stacked_bar('family')

## 6. Daftar output

In [ ]:
for p in sorted(RESULT_DIR.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(BASE_DIR)):55s} {p.stat().st_size/1024:9.1f} KB")